# Esame 22 luglio 2025

Dato un dataset per il clustering retail con tanti scontrini su kaggle con qualche migliaio di osservazioni, mettere a confronto misture di gaussiane nell'implementazione standard e quella bayesiana e un altro modello di clustering a scelta come KMeans.


In [9]:
from sklearn.base import clone
from sklearn.metrics import silhouette_score
from sklearn.model_selection import ParameterGrid, RepeatedStratifiedKFold
from sklearn.cluster import KMeans
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

# pseudo-stratified labels con KMeans, etichette fittizie per stratificazione
def generate_stratified_bins(X, n_bins=3):
    km = KMeans(n_clusters=n_bins, random_state=42, n_init= 10)
    return km.fit_predict(X)


def clustering_cv(X, model_class, param_grid, n_folds=5, n_repeats=5,
                  stratify_bins=5,
                  sample_frac=0.3,
                  random_state=42):
    results = []
    X = StandardScaler().fit_transform(X)

    stratify_labels = generate_stratified_bins(X, n_bins=stratify_bins)
    rskf = RepeatedStratifiedKFold(n_splits=n_folds, n_repeats=n_repeats, random_state=random_state)

    for params in ParameterGrid(param_grid):
        for fold_idx, (train_idx, test_idx) in enumerate(rskf.split(X, stratify_labels)):
            X_train = X[train_idx]
            X_train_sampled = resample(X_train, replace=False, n_samples=int(sample_frac * len(X_train)),
                                       random_state=random_state)

            model = clone(model_class(**params))
            model.fit(X_train_sampled)

            if hasattr(model, "labels_"):
                labels = model.labels_
                inertia = model.inertia_ if hasattr(model, "inertia_") else np.nan
            elif hasattr(model, "predict"):
                labels = model.predict(X_train_sampled)
                inertia = np.nan
            else:
                continue

            if len(set(labels)) > 1:
                sil = silhouette_score(X_train_sampled, labels)
            else:
                sil = np.nan

            results.append({
                "params": params,
                "fold": fold_idx,
                "silhouette": sil,
                "inertia": inertia,
                "n_clusters": len(set(labels))
            })

    df = pd.DataFrame(results)
    mean_scores = df.groupby(df['params'].apply(lambda x: tuple(sorted(x.items())))).mean(numeric_only=True)
    best_idx = mean_scores["silhouette"].idxmax()
    best_params = dict(best_idx)

    return best_params, df

# Dataset

In [2]:
import kagglehub

path = kagglehub.dataset_download("gabrielramos87/an-online-shop-business")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\rubin\.cache\kagglehub\datasets\gabrielramos87\an-online-shop-business\versions\7


In [3]:
import os
import pandas as pd

dataset_path = "C:/Users/rubin/.cache/kagglehub/datasets/gabrielramos87/an-online-shop-business/versions/7"

files = os.listdir(dataset_path)
print("Files in dataset:", files)

Files in dataset: ['Sales Transaction v.4a.csv']


In [4]:
csv_file_path = os.path.join(dataset_path, "Sales Transaction v.4a.csv")

df_kaggle = pd.read_csv(csv_file_path)
print(df_kaggle.head())

# conto i valori unici nella colonna 'Country'
print("Valori unici in 'Country':", df_kaggle['Country'].nunique())

  TransactionNo       Date ProductNo                          ProductName  \
0        581482  12/9/2019     22485        Set Of 2 Wooden Market Crates   
1        581475  12/9/2019     22596  Christmas Star Wish List Chalkboard   
2        581475  12/9/2019     23235             Storage Tin Vintage Leaf   
3        581475  12/9/2019     23272    Tree T-Light Holder Willie Winkie   
4        581475  12/9/2019     23239    Set Of 4 Knick Knack Tins Poppies   

   Price  Quantity  CustomerNo         Country  
0  21.47        12     17490.0  United Kingdom  
1  10.65        36     13069.0  United Kingdom  
2  11.53        12     13069.0  United Kingdom  
3  10.65        12     13069.0  United Kingdom  
4  11.94         6     13069.0  United Kingdom  
Valori unici in 'Country': 38


## Dataset splitting

In [5]:
X = df_kaggle.drop(columns=['Country'])
y = df_kaggle['Country']

In [6]:
import numpy as np

X = df_kaggle.drop(columns=['Country']).select_dtypes(include=[np.number])
X = X[~np.isnan(X).any(axis=1)]

# Uso soltanto le prime 1000 osservazioni per velocizzare il processo
X = X.sample(n=1000, random_state=42)

In [7]:
from sklearn.mixture import GaussianMixture

param_grid = {
    'n_components': [2, 3, 4], # Numero di componenti / cluster
    'covariance_type': ['full', 'diag'], # Tipi di matrice di covarianza (full, tied, diag, spherical)
    'max_iter': [100, 120] # Iterazioni massime durante il fit
}

In [10]:
best_params, results_df_gmm = clustering_cv(
    X=X,
    model_class=GaussianMixture,
    param_grid=param_grid,
    n_folds=5,# Numero di fold per la cross-validation
    n_repeats=2, # Numero di ripetizioni della cross-validation
    stratify_bins=5, # Numero di bin per la stratificazione (etichette fittizie)
    sample_frac=0.5,
    random_state=42
)

C:\Users\rubin\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(
C:\Users\rubin\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
C:\Users\rubin\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(
C:\Users\rubin\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the

In [14]:
print("Migliori parametri:", best_params)

Migliori parametri: {'n_clusters': 2}


## Clustering con kMedoids

In [12]:
from sklearn_extra.cluster import KMedoids

param_grid = {
    'n_clusters': [2, 3],
}

best_params, results_df = clustering_cv(X, KMedoids, param_grid)
print("Best parameters:", best_params)

C:\Users\rubin\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1446: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
  warnings.warn(
C:\Users\rubin\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
C:\Users\rubin\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
C:\Users\rubin\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
C:\Users\rubin\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has 

Best parameters: {'n_clusters': 2}


C:\Users\rubin\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
C:\Users\rubin\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
C:\Users\rubin\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
C:\Users\rubin\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
C:\Users\rubin\anaconda3\Lib\site-packages\sklearn\model_selection\_split.py:737: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
